<a href="https://colab.research.google.com/github/0301sdh/faceguard/blob/main/notebooks/simswap_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a simple example of SimSwap on processing video with multiple faces. You can change the codes for inference based on our other scripts for image or single face swapping.

Code path: https://github.com/neuralchen/SimSwap

Paper path: https://arxiv.org/pdf/2106.06340v1.pdf or https://dl.acm.org/doi/10.1145/3394171.3413630

In [9]:
## make sure you are using a runtime with GPU
## you can check at Runtime/Change runtime type in the top bar.
!nvidia-smi

Thu Oct  1 11:38:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             31W /   70W |    1245MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Installation

All file changes made by this notebook are temporary.
You can try to mount your own google drive to store files if you want.


In [10]:
!git clone https://github.com/neuralchen/SimSwap
!cd SimSwap && git pull

Cloning into 'SimSwap'...
remote: Enumerating objects: 1141, done.
remote: Counting objects: 100% (508/508), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 1141 (delta 419), reused 382 (delta 382), pack-reused 633 (from 1)
Receiving objects: 100% (1141/1141), 211.46 MiB | 41.71 MiB/s, done.
Resolving deltas: 100% (601/601), done.
Already up to date.


In [1]:
!pip install insightface==0.2.1 onnxruntime moviepy
!pip install googledrivedownloader
!pip install imageio==2.4.1

  Using cached imageio-2.38.0-py3-none-any.whl.metadata (9.7 kB)
Using cached imageio-2.38.0-py3-none-any.whl (328 kB)
  Attempting uninstall: imageio
    Found existing installation: imageio 2.4.1
    Uninstalling imageio-2.4.1:
      Successfully uninstalled imageio-2.4.1
  Using cached imageio-2.4.1-py3-none-any.whl
  Attempting uninstall: imageio
    Found existing installation: ImageIO 2.38.0
    Uninstalling ImageIO-2.38.0:
      Successfully uninstalled ImageIO-2.38.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moviepy 1.0.3 requires imageio<3.0,>=2.5; python_version >= "3.4", but you have imageio 2.4.1 which is incompatible.
scikit-image 0.25.2 requires imageio!=2.35.0,>=2.33, but you have imageio 2.4.1 which is incompatible.


In [2]:
import os
os.chdir("SimSwap")
!ls

 antelope.zip	       predict.py
 arcface_model	       README.md
 checkpoints	       SimSwap
 checkpoints.zip      'SimSwap colab.ipynb'
 cog.yaml	       simswaplogo
 crop_224	       test_one_image.py
 data		       test_video_swapmulti.py
 demo_file	       test_video_swap_multispecific.py
 docs		       test_video_swapsingle.py
 download-weights.sh   test_video_swapspecific.py
 insightface_func      test_wholeimage_swapmulti.py
 LICENSE	       test_wholeimage_swap_multispecific.py
 models		       test_wholeimage_swapsingle.py
 MultiSpecific.ipynb   test_wholeimage_swapspecific.py
 options	       tmp
 output		       train.ipynb
 parsing_model	       train.py
 pg_modules	       util


In [3]:
#from google_drive_downloader import GoogleDriveDownloader

### it seems that google drive link may not be permenant, you can find this ID from our open url.
# GoogleDriveDownloader.download_file_from_google_drive(file_id='1TLNdIufzwesDbyr_nVTR7Zrx9oRHLM_N',
#                                     dest_path='./arcface_model/arcface_checkpoint.tar')
# GoogleDriveDownloader.download_file_from_google_drive(file_id='1PXkRiBUYbu1xWpQyDEJvGKeqqUFthJcI',
#                                     dest_path='./checkpoints.zip')

!wget -P ./arcface_model https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
!wget https://github.com/neuralchen/SimSwap/releases/download/1.0/checkpoints.zip
!unzip ./checkpoints.zip  -d ./checkpoints
!wget -P ./parsing_model/checkpoint https://github.com/neuralchen/SimSwap/releases/download/1.0/79999_iter.pth

--2026-10-01 11:40:41--  https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/374891081/f01468b3-446b-4867-8c78-6d496183f9e6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-01T12%3A19%3A53Z&rscd=attachment%3B+filename%3Darcface_checkpoint.tar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-01T11%3A19%3A30Z&ske=2026-10-01T12%3A19%3A53Z&sks=b&skv=2018-11-09&sig=gWfn%2BoIA42tFK60un4j34%2FzKtTbFuSRO0i5VbudVfOY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MDg1ODQ0MSwibmJmIjoxNzkwODU0ODQxLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJv

In [4]:
## You can upload filed manually
# from google.colab import drive
# drive.mount('/content/gdrive')

### Now onedrive file can be downloaded in Colab directly!
### If the link blow is not permanent, you can just download it from the
### open url(can be found at [our repo]/doc/guidance/preparation.md) and copy the assigned download link here.
### many thanks to woctezuma for this very useful help
!wget --no-check-certificate "https://sh23tw.dm.files.1drv.com/y4mmGiIkNVigkSwOKDcV3nwMJulRGhbtHdkheehR5TArc52UjudUYNXAEvKCii2O5LAmzGCGK6IfleocxuDeoKxDZkNzDRSt4ZUlEt8GlSOpCXAFEkBwaZimtWGDRbpIGpb_pz9Nq5jATBQpezBS6G_UtspWTkgrXHHxhviV2nWy8APPx134zOZrUIbkSF6xnsqzs3uZ_SEX_m9Rey0ykpx9w" -O antelope.zip
!unzip ./antelope.zip -d ./insightface_func/models/


--2026-10-01 11:41:28--  https://sh23tw.dm.files.1drv.com/y4mmGiIkNVigkSwOKDcV3nwMJulRGhbtHdkheehR5TArc52UjudUYNXAEvKCii2O5LAmzGCGK6IfleocxuDeoKxDZkNzDRSt4ZUlEt8GlSOpCXAFEkBwaZimtWGDRbpIGpb_pz9Nq5jATBQpezBS6G_UtspWTkgrXHHxhviV2nWy8APPx134zOZrUIbkSF6xnsqzs3uZ_SEX_m9Rey0ykpx9w
Resolving sh23tw.dm.files.1drv.com (sh23tw.dm.files.1drv.com)... 150.171.22.11, 2620:1ec:50::11
Connecting to sh23tw.dm.files.1drv.com (sh23tw.dm.files.1drv.com)|150.171.22.11|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-10-01 11:41:28 ERROR 404: Not Found.

Archive:  ./antelope.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ./antelope.zip or
        ./antelope.zip.zip, and cannot find ./antelope.zip.ZIP, period.


## Inference

In [9]:
import cv2
import torch
import fractions
import numpy as np
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms
from models.models import create_model
from options.test_options import TestOptions
from insightface_func.face_detect_crop_multi import Face_detect_crop
from util.videoswap import video_swap
from util.add_watermark import watermark_image

In [10]:
transformer = transforms.Compose([
        transforms.ToTensor(),
        #transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

transformer_Arcface = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

detransformer = transforms.Compose([
        transforms.Normalize([0, 0, 0], [1/0.229, 1/0.224, 1/0.225]),
        transforms.Normalize([-0.485, -0.456, -0.406], [1, 1, 1])
    ])

In [7]:
!sed -i 's/map_location=torch.device("cpu"))/map_location=torch.device("cpu"), weights_only=False)/' /content/SimSwap/models/fs_model.py
!grep -n "torch.load" /content/SimSwap/models/fs_model.py

64:        netArc_checkpoint = torch.load(netArc_checkpoint, map_location=torch.device("cpu"), weights_only=False)


In [8]:
!wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip -O /content/antelopev2.zip
!unzip -o -q /content/antelopev2.zip -d /content/antelopev2_tmp
!mkdir -p /content/SimSwap/insightface_func/models/antelope
!cp $(find /content/antelopev2_tmp -name scrfd_10g_bnkps.onnx) /content/SimSwap/insightface_func/models/antelope/
!ls -la /content/SimSwap/insightface_func/models/antelope

total 16536
drwxr-xr-x 2 root root     4096 Oct  1 10:50 .
drwxr-xr-x 3 root root     4096 Oct  1 10:50 ..
-rw-r--r-- 1 root root 16923827 Oct  1 11:41 scrfd_10g_bnkps.onnx


In [11]:
import numpy as np
np.float = float   # 옛 코드의 np.float를 Python 기본 float로 연결
np.int = int       # 같은 이유로 없어진 np.int도 미리 연결 (다음 오류 예방)

In [12]:
opt = TestOptions()
opt.initialize()
opt.parser.add_argument('-f') ## dummy arg to avoid bug
opt = opt.parse()
opt.pic_a_path = './demo_file/Iron_man.jpg' ## or replace it with image from your own google drive
opt.video_path = './demo_file/multi_people_1080p.mp4' ## or replace it with video from your own google drive
opt.output_path = './output/demo.mp4'
opt.temp_path = './tmp'
opt.Arc_path = './arcface_model/arcface_checkpoint.tar'
opt.isTrain = False
opt.use_mask = True  ## new feature up-to-date
opt.crop_size = 224  ## 받은 가중치(people)가 224용이라 맞춰 줌

crop_size = opt.crop_size

torch.nn.Module.dump_patches = True
model = create_model(opt)
model.eval()

app = Face_detect_crop(name='antelope', root='./insightface_func/models')
app.prepare(ctx_id= 0, det_thresh=0.6, det_size=(640,640))

with torch.no_grad():
    pic_a = opt.pic_a_path
    # img_a = Image.open(pic_a).convert('RGB')
    img_a_whole = cv2.imread(pic_a)
    img_a_align_crop, _ = app.get(img_a_whole,crop_size)
    img_a_align_crop_pil = Image.fromarray(cv2.cvtColor(img_a_align_crop[0],cv2.COLOR_BGR2RGB))
    img_a = transformer_Arcface(img_a_align_crop_pil)
    img_id = img_a.view(-1, img_a.shape[0], img_a.shape[1], img_a.shape[2])

    # convert numpy to tensor
    img_id = img_id.cuda()

    #create latent id
    img_id_downsample = F.interpolate(img_id, size=(112,112))
    latend_id = model.netArc(img_id_downsample)
    latend_id = latend_id.detach().to('cpu')
    latend_id = latend_id/np.linalg.norm(latend_id,axis=1,keepdims=True)
    latend_id = latend_id.to('cuda')

    #video_swap(opt.video_path, latend_id, model, app, opt.output_path, temp_results_dir=opt.temp_path, use_mask=opt.use_mask)

------------ Options -------------
Arc_path: arcface_model/arcface_checkpoint.tar
aspect_ratio: 1.0
batchSize: 8
checkpoints_dir: ./checkpoints
cluster_path: features_clustered_010.npy
crop_size: 512
data_type: 32
dataroot: ./datasets/cityscapes/
display_winsize: 512
engine: None
export_onnx: None
f: /root/.local/share/jupyter/runtime/kernel-5b856511-cdb4-437f-bf49-4d3ab654fb04.json
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
how_many: 50
id_thres: 0.03
image_size: 224
input_nc: 3
instance_feat: False
isTrain: False
label_feat: False
label_nc: 0
latent_size: 512
loadSize: 1024
load_features: False
local_rank: 0
max_dataset_size: inf
multisepcific_dir: ./demo_file/multispecific
nThreads: 2
n_blocks_global: 6
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 3
n_local_enhancers: 1
name: people
nef: 16
netG: global
ngf: 64
niter_fix_global: 0
no_flip: False
no_instance: False
no_simswaplogo: False
norm: batch
norm_G: spectralspadesyncbatch3x3
ntest: inf
onnx: 

  latend_id = latend_id/np.linalg.norm(latend_id,axis=1,keepdims=True)



In [21]:
import os
from util.reverse2original import reverse2wholeimage
from util.norm import SpecificNorm
from parsing_model.model import BiSeNet

# 1) 사진 경로
source_path = '/content/my_face3.jpg'        # 내 사진 (누구 얼굴로 만들 것인가)
target_path = './demo_file/specific1.png'   # 다른 사람 사진 (어디에 붙일 것인가)
output_path = './output/baseline_swap3.jpg'
os.makedirs('./output', exist_ok=True)

# 2) 합성한 얼굴을 자연스럽게 붙이는 parsing 모델
net = BiSeNet(n_classes=19).cuda()
net.load_state_dict(torch.load('./parsing_model/checkpoint/79999_iter.pth'))
net.eval()
spNorm = SpecificNorm()

with torch.no_grad():
    # 3) source → 명세서
    img_a_whole = cv2.imread(source_path)
    img_a_align_crop, _ = app.get(img_a_whole, crop_size)
    img_a = transformer_Arcface(Image.fromarray(cv2.cvtColor(img_a_align_crop[0], cv2.COLOR_BGR2RGB)))
    img_id = img_a.unsqueeze(0).cuda()
    latend_id = model.netArc(F.interpolate(img_id, size=(112, 112)))
    latend_id = F.normalize(latend_id, p=2, dim=1)

    # 4) target 얼굴을 명세서대로 다시 그리기
    img_b_whole = cv2.imread(target_path)
    img_b_align_crop_list, b_mat_list = app.get(img_b_whole, crop_size)
    swap_result_list, b_crop_tensor_list = [], []
    for b_crop in img_b_align_crop_list:
        b_tensor = torch.from_numpy(cv2.cvtColor(b_crop, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255)[None].cuda()
        swap_result_list.append(model(None, b_tensor, latend_id, None, True)[0])
        b_crop_tensor_list.append(b_tensor)

    # 5) 원래 target 사진에 붙여서 저장
    reverse2wholeimage(b_crop_tensor_list, swap_result_list, b_mat_list, crop_size, img_b_whole, None,
                       output_path, no_simswaplogo=True, pasring_model=net, use_mask=True, norm=spNorm)

print('저장 완료:', output_path)

저장 완료: ./output/baseline_swap3.jpg


In [20]:
def arc_embed(img_bgr):
    # SimSwap과 같은 방식: 얼굴 찾기·정렬 → 정규화 → 112×112 → ArcFace → 길이 1로 맞춤
    crops, _ = app.get(img_bgr, crop_size)
    x = transformer_Arcface(Image.fromarray(cv2.cvtColor(crops[0], cv2.COLOR_BGR2RGB))).unsqueeze(0).cuda()
    with torch.no_grad():
        e = model.netArc(F.interpolate(x, size=(112, 112)))
    return F.normalize(e, p=2, dim=1), crops[0]

src_emb, src_crop = arc_embed(cv2.imread(source_path))   # 내 사진
tgt_emb, _        = arc_embed(cv2.imread(target_path))   # target 원본
res_emb, res_crop = arc_embed(cv2.imread(output_path))   # 합성 결과

# 정렬된 얼굴을 저장해서 눈으로 확인
cv2.imwrite('./output/source_crop.jpg', src_crop)
cv2.imwrite('./output/result_crop.jpg', res_crop)

# 길이가 1인 벡터끼리 곱해서 더하면 코사인 유사도
print('결과 vs 내 사진    :', (res_emb * src_emb).sum().item())
print('결과 vs target     :', (res_emb * tgt_emb).sum().item())
print('내 사진 vs target  :', (src_emb * tgt_emb).sum().item())

결과 vs 내 사진    : 0.6875535249710083
결과 vs target     : 0.3711407780647278
내 사진 vs target  : 0.06924822181463242
